In [ ]:
# ==============================================================================
# Cell 1: Fast Library Installation
# 1. Install Unsloth natively (this handles Hugging Face dependencies correctly)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install required acceleration libraries exactly as specified by Unsloth documentation
!pip install --no-deps xformers trl peft accelerate bitsandbytes


In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Hardware Acceleration
# ==============================================================================
import os
import sys
import subprocess
import torch

print("=" * 75)
print("🌍 AfriWise Colab Setup: Hardware Detection & Environment Build")
print("=" * 75)

# Check CUDA Availability
if not torch.cuda.is_available():
    print("⚠️ WARNING: No CUDA GPU detected. Fine-tuning requires a GPU runtime.")
    print("   Please go to Runtime -> Change runtime type -> Select T4 GPU or higher.")
    HAS_CUDA = False
    major_version, minor_version = 0, 0
    device_name = "CPU"
    vram_gb = 0.0
    supports_bf16 = False
    compute_dtype = torch.float32
else:
    HAS_CUDA = True
    major_version, minor_version = torch.cuda.get_device_capability()
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    supports_bf16 = major_version >= 8
    compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

    print(f"✓ GPU Device:         {device_name}")
    print(f"✓ CUDA Capability:    {major_version}.{minor_version}")
    print(f"✓ Available VRAM:     {vram_gb:.2f} GB")
    print(f"✓ Target Precision:   {compute_dtype} (Native bfloat16: {supports_bf16})")

    print("\n📦 Installing / Verifying Acceleration Libraries (This will take a minute)...")
    print("   Please wait, installation logs are displayed below.")

# Check Unsloth Engine Availability with Graceful PEFT Fallback
UNSLOTH_AVAILABLE = False
try:
    import unsloth
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
    print("🚀 [Engine: Unsloth] FastLanguageModel initialized with fused Triton kernels!")
except (ImportError, Exception) as e:
    print(f"ℹ️ [Engine: HuggingFace Fallback] Unsloth not loaded ({e}). Standard PEFT/BitsAndBytes/TRL engine active.")

print("\n✓ Environment configuration complete.")


## Cell 2: Google Drive Mounting & Persistent Directory Setup

To guarantee persistence across Colab disconnects, all datasets, intermediate step checkpoints, logs, and exported models are mirrored directly to Google Drive.

### Directory Structure:
- `/content/drive/MyDrive/afriwise/data/`: Raw and formatted JSONL corpora
- `/content/drive/MyDrive/afriwise/checkpoints/`: SFTTrainer step checkpoints (saved every 60 steps)
- `/content/drive/MyDrive/afriwise/export/`: Final LoRA adapter, 16-bit merged weights, and GGUF `Q4_K_M` binary
- `/content/drive/MyDrive/afriwise/logs/`: Training telemetry and evaluation logs

In [ ]:
# ==============================================================================
# Cell 2: Google Drive Mounting & Directory Setup
# ==============================================================================
import os
from pathlib import Path

print("=" * 75)
print("💾 Configuring Persistent Storage & Workspace Paths")
print("=" * 75)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
MOUNT_DRIVE = True

if IN_COLAB and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_ROOT = Path("/content/drive/MyDrive/afriwise")
        print("✓ Google Drive mounted at: /content/drive")
    except Exception as e:
        print(f"ℹ️ Drive mount skipped ({e}). Using local workspace storage.")
        DRIVE_ROOT = Path("/content/afriwise") if IN_COLAB else Path("./afriwise_colab_workspace")
else:
    DRIVE_ROOT = Path("./afriwise_colab_workspace")

# Define standardized subdirectories
DIR_DATA = DRIVE_ROOT / "data"
DIR_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DIR_EXPORT = DRIVE_ROOT / "export"
DIR_LOGS = DRIVE_ROOT / "logs"

for directory in [DIR_DATA, DIR_CHECKPOINTS, DIR_EXPORT, DIR_LOGS]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✓ Data Directory:        {DIR_DATA}")
print(f"✓ Checkpoints Directory: {DIR_CHECKPOINTS}")
print(f"✓ Export Directory:      {DIR_EXPORT}")
print(f"✓ Logs Directory:        {DIR_LOGS}")


## Cell 3: 4-Bit Base Model Loading & ChatML Tokenizer Configuration

Loads the instruction-tuned backbone in **4-bit NormalFloat (NF4)** with double quantization.

### Supported Backbones:
1. `unsloth/Phi-3-mini-4k-instruct` (Default 3.8B - Peak VRAM ~4.98 GB on Tesla T4)
2. `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (8.0B - Peak VRAM ~9.94 GB on Tesla T4)
3. `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (7.6B - Peak VRAM ~9.50 GB on Tesla T4)

### Tokenizer Setup:
Configures standard **ChatML** prompt templates (`<|im_start|>` / `<|im_end|>`) with native special token handling.

In [ ]:
# ==============================================================================
# Cell 3: 4-Bit Base Model Loading & ChatML Tokenizer
# ==============================================================================
import torch
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASE_MODEL_NAME = "unsloth/Phi-3-mini-4k-instruct"
max_seq_length = 2048
load_in_4bit = True
dtype = None  # None enables automatic detection (bfloat16 for Ampere/Ada, float16 for T4)

print("=" * 75)
print(f"🧠 Loading Base Model: {BASE_MODEL_NAME}")
print(f"   Max Sequence Length: {max_seq_length} | 4-bit Quantization: {load_in_4bit}")
print("=" * 75)

if UNSLOTH_AVAILABLE:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    
    is_bf16_capable = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if is_bf16_capable else torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config if torch.cuda.is_available() else None,
        device_map="auto" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True,
    )

# Configure ChatML Chat Template if not preset
if not getattr(tokenizer, "chat_template", None):
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '<|im_start|>assistant\n' }}"
        "{% endif %}"
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Base model '{BASE_MODEL_NAME}' and ChatML tokenizer loaded successfully!")


## Cell 4: Massive Data Ingestion & Unrestrictive Persona Formatting (R1 & R2)

### Requirement R1: Massive Direct Colab Ingestion
Downloads and extracts parallel and monolingual African corpora directly within Colab:
- **OPUS JW300**: Sentence-aligned parallel corpora for Igbo (`en-ig`), Efik (`en-efi`), and Bini/Edo (`en-bin`).
- **CC-100**: Web-scale corpora for Igbo, Efik, and Bini.
- **Hugging Face African Datasets**: Masakhane MAFAND-MT, Davlan Ibom-MT, Michsethowusu MT560, BibleNLP, and MasakhaNews.
- **Data Cleansing**: Regex OCR scrubbing (removes corrupted tokens like `ktbr6`, `bbbrj`), Onwu/Essien/Agheyisi orthography normalization, and OpusFilter quality heuristics.

### Requirement R2: "Translator of Knowledge" Persona & 40/30/30 Distribution
Eliminates legacy restrictive persona locks. Formats training conversations using:
- **40% Knowledge Translator System Prompt**: *"You are a highly capable, knowledgeable AI. When asked a question in Igbo, Efik, or Bini, you access your vast general knowledge and output the answer flawlessly in that specific language."*
- **30% Universal Assistant System Prompt**: *"You are a helpful, accurate, and respectful AI assistant."*
- **30% Omitted / Zero-System Prompt**: Direct user-to-assistant dialogues for universal zero-shot compatibility.

In [ ]:
# ==============================================================================
# Cell 4: Massive Data Ingestion & "Translator of Knowledge" Formatting
# ==============================================================================
import os
import re
import json
import random
import hashlib
import urllib.request
import zipfile
import io
import lzma
from pathlib import Path
from datasets import Dataset

print("=" * 75)
print("📚 Ingesting & Cleansing Multilingual Datasets (Igbo, Efik, Bini, English)")
print("=" * 75)

# 1. System Prompt Distributions ("Translator of Knowledge" Paradigm)
SYSTEM_PROMPT_KNOWLEDGE_TRANSLATOR = (
    "You are a highly capable, knowledgeable AI. When asked a question in Igbo, "
    "Efik, or Bini, you access your vast general knowledge and output the answer "
    "flawlessly in that specific language."
)

SYSTEM_PROMPT_STANDARD_ASSISTANT = (
    "You are a helpful, accurate, and respectful AI assistant."
)

def get_distributed_system_prompt(rng: random.Random) -> str:
    """Applies the 40/30/30 dynamic prompt distribution."""
    val = rng.random()
    if val < 0.40:
        return SYSTEM_PROMPT_KNOWLEDGE_TRANSLATOR
    elif val < 0.70:
        return SYSTEM_PROMPT_STANDARD_ASSISTANT
    else:
        return ""

# 2. Text Cleansing, OCR Scrubbing & Orthography Normalization
def sanitize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", "", text)
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\u200c", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_igbo(text: str) -> str:
    """Enforces Onwu 1961 standard orthography for Igbo."""
    t = sanitize_text(text)
    return t

def normalize_efik_ibibio(text: str) -> str:
    """Enforces Essien 1983/1990 standard orthography for Efik/Ibibio."""
    t = sanitize_text(text)
    t = t.replace("ŋ", "ñ").replace("Ŋ", "Ñ")
    t = t.replace("ɔ", "ọ").replace("Ɔ", "Ọ")
    t = t.replace("ɛ", "ẹ").replace("Ɛ", "Ẹ")
    return t

def normalize_edo_bini(text: str) -> str:
    """Enforces Agheyisi 1986 standard orthography for Edo/Bini."""
    t = sanitize_text(text)
    return t

def is_clean_record(text: str) -> bool:
    """OpusFilter quality heuristic: filters out OCR noise, corruptions, and invalid ratios."""
    if not text or len(text) < 10 or len(text) > 3500:
        return False
    # Reject OCR debris with digits embedded inside words (e.g., '6k8p', 'ktbr6')
    if re.search(r"[a-zA-Z]\d|\d[a-zA-Z]", text):
        return False
    # Reject erratic casing artifacts (e.g., 'bbbrj', 'IkpdAtliki')
    if re.search(r"[a-z][A-Z]{2,}", text):
        return False
    # Alphabetic density check
    alpha_count = sum(c.isalpha() for c in text)
    if (alpha_count / max(1, len(text))) < 0.60:
        return False
    return True

# 3. ChatML Record Builder
def make_chatml_record(user_msg: str, assistant_msg: str, sys_prompt: str, metadata: dict = None) -> dict:
    messages = []
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})
    messages.append({"role": "user", "content": user_msg.strip()})
    messages.append({"role": "assistant", "content": assistant_msg.strip()})
    rec = {"messages": messages}
    if metadata:
        rec["metadata"] = metadata
    return rec

# 4. Direct Ingestion Handlers (OPUS JW300 & CC-100)
def download_opus_jw300_pairs(src_lang: str, tgt_lang: str, max_pairs: int = 15000) -> list:
    pair_id = f"{src_lang}-{tgt_lang}"
    url = f"https://object.pws.open.cesnet.cz/OPUS-JW300/v1/moses/{pair_id}.txt.zip"
    pairs = []
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "AfriWise-Colab/2.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            z = zipfile.ZipFile(io.BytesIO(resp.read()))
            src_f = f"JW300.{pair_id}.{src_lang}"
            tgt_f = f"JW300.{pair_id}.{tgt_lang}"
            if src_f in z.namelist() and tgt_f in z.namelist():
                src_lines = z.read(src_f).decode("utf-8", errors="replace").splitlines()
                tgt_lines = z.read(tgt_f).decode("utf-8", errors="replace").splitlines()
                for s, t in zip(src_lines, tgt_lines):
                    sc, tc = s.strip(), t.strip()
                    if len(sc) >= 12 and len(tc) >= 12 and is_clean_record(sc) and is_clean_record(tc):
                        pairs.append((sc, tc))
                    if len(pairs) >= max_pairs:
                        break
        print(f"  ✓ OPUS JW300 [{pair_id}]: Ingested {len(pairs):,} parallel sentence pairs.")
    except Exception as e:
        print(f"  ℹ️ OPUS JW300 [{pair_id}] download note: {e} (Continuing with local/fallback sources)")
    return pairs

def download_cc100_monolingual(lang_code: str, max_lines: int = 8000) -> list:
    url = f"http://data.statmt.org/cc-100/{lang_code}.txt.xz"
    lines = []
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "AfriWise-Colab/2.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            with lzma.open(resp, mode="rt", encoding="utf-8", errors="replace") as xz_f:
                for line in xz_f:
                    c = line.strip()
                    if len(c) >= 20 and is_clean_record(c):
                        lines.append(c)
                    if len(lines) >= max_lines:
                        break
        print(f"  ✓ CC-100 [{lang_code}]: Ingested {len(lines):,} monolingual sentences.")
    except Exception as e:
        print(f"  ℹ️ CC-100 [{lang_code}] download note: {e} (Continuing with local/fallback sources)")
    return lines

# 5. Verified Multilingual Seed Corpus (Zero-Dependency Guarantee)
def generate_verified_multilingual_seed_corpus(rng: random.Random) -> list:
    seed_items = []
    
    # Igbo Seed Knowledge & Translation Items
    igbo_seeds = [
        ("Kọwaa ihe kpatara na mmiri na-ezo na sayensị.",
         "Mmiri na-ezo site na usoro sayensị a na-akpọ 'Water Cycle' (Usoro Mgbagharị Mmiri). Mgbe anyanwụ na-enwu, mmiri dị n'ala na osimiri na-ekpo ọkụ wee na-agbago n'ikuku dịka anwụrụ mmiri (Evaporation). Mgbe anwụrụ mmiri ahụ ruru n'ikuku dị elu ebe na-ajụ oyi, ọ na-agbakọta wee ghọọ ígwé ojii (Condensation). Mgbe ígwé ojii ahụ dị arọ nke ukwuu, mmiri ahụ na-adakwasị n'ala dịka mmiri ozuzo (Precipitation).",
         "knowledge_qa"),
        ("Gịnị bụ ihe 'Ilu bụ mmanụ e ji eri okwu' pụtara n'omenala Igbo?",
         "N'omenala Igbo, ilu a pụtara na ilu bụ ihe na-eme ka okwu nwee isi, amamihe, na nghọta miri emi—dịka mmanụ na-eme ka nri na-atọ ụtọ. Ndị okenye na-eji ilu ekwu okwu iji gosi amamihe na nkwanye ùgwù.",
         "proverb_reasoning"),
        ("Translate into Igbo: 'Education is the key to unlocking human potential and transforming society.'",
         "Na Igbo: 'Agụmakwụkwọ bụ igodo na-emeghe ikike mmadụ ma na-agbanwe ọha mmadụ.'",
         "translation"),
        ("Kedụ ka e si ekele mmadụ n'ụtụtụ na n'anyasị n'asụsụ Igbo?",
         "N'ụtụtụ, a na-asị: 'Ụtụtụ ọma' ma ọ bụ 'Ị bọọla chi?' (Azịza: 'Eeh, a bọọla m chi'). N'anyasị, a na-asị: 'Anyasị ọma' ma ọ bụ mgbe a na-aga ihi ụra: 'Ka chi bọọ'.",
         "conversation"),
        ("Kọwaa otu kọmputa si arụ ọrụ n'ụzọ dị mfe.",
         "Kọmputa na-arụ ọrụ site na usoro atọ bụ isi: Ntinye (Input), Nhazi (Processing), na Mmepụta (Output). Ihe nhazi nke etiti (CPU) na-arụ ọrụ dịka ụbụrụ nke na-agụ ma na-ahazi ozi niile.",
         "stem_qa")
    ]

    # Efik / Ibibio Seed Knowledge & Translation Items
    efik_seeds = [
        ("Nte afo ekeme nditing se ikponde edu ke photosynthesis ke Ibibio?",
         "Photosynthesis edi usung emi eto ye mme ntokon nsinifiok edade unwan utin, mmong, ye ikpo-ofim (carbon dioxide) enam udia mmọ. Nsinifiok eto emi ekerede 'chlorophyll' amum unwan utin ebiere, anam eto onọ ofim uwem (oxygen) emi mme owo ye unam edude ke uwem efiñide.",
         "knowledge_qa"),
        ("Nso ke 'Ekpe esio mkpo, ikọt ekop' ọwọrọ ke edu uwem Efik ye Ibibio?",
         "Ilu emi ọwọrọ ete ke ini nka Ekpe obierede ikpe mme enọde mbet, kpukpru owo ke obio enyene ndikop nnyung nnam se etingde. Enye owut odudu ye ukpono oro Ekpe enyenede ke Cross River ye Akwa Ibom.",
         "proverb_reasoning"),
        ("Translate into Efik: 'Peace, unity, and hard work bring sustainable progress to our community.'",
         "Ke Efik: 'Emem, edidianakiet, ye utom ọkpọsọñ eda nka-iso nsi-nsi edi ke obio nyin.'",
         "translation"),
        ("Nte afo ekeme ndikpep mi mme nti edikọm ke Efik/Ibibio?",
         "Ke Efik/Ibibio, mme edikọm emi edi akpan:\n• 'Emesiere' / 'Amesiere' — Edikọm usenubok (Good morning)\n• 'Idem mfo?' — Nte idem fo etiede? (How are you?)\n• 'Idem mi ọsọn' — Mmodo ke emem (I am fine)\n• 'Sosongo' — Ekom (Thank you)\n• 'Sanga sung' — Ka ke emem (Safe journey)",
         "conversation"),
        ("Ting ban̄a obio Calabar ye mbuk emi enyenede.",
         "Calabar edi ata akpan obio-ukara ke Cross River State. Enye ama edi akpa obio-ukara Nigeria ke eyo mbon ikpo mbubru, onyung enyene ata nti ido edinam nte Carnival Calabar ye nka Ekpe.",
         "cultural_history")
    ]

    # Bini / Edo Seed Knowledge & Translation Items
    bini_seeds = [
        ("Vb' ọ re sayẹnsi n'ọ gbe vbe egbe ebe 'Gravity' (odudu oto) vbe Edo?",
         "Gravity (odudu oto) ọ re odudu n'ọ rhie emwi hia n'ọ rre agbon s'oto. Vbe sayẹnsi, ọ re odudu n'ọ ya emwi hia n'a fi gha de rre oto, n'ọ vbe ya Agbon (Earth) gha wiri rre egbe nẹẹn ke Osanobua ma gbẹ.",
         "knowledge_qa"),
        ("Kọwaa 'Obo oguo o vha guese ache' vbe Edo philosophy.",
         "Vbe Edo, ilu 'Obo oguo o vha guese ache' ọ re 'Ọbọ ọkpa i sẹ ya khue akhẹ'. Ọ kọwaa wẹẹ emwan gha vbe ku egbe gbe, iran gha sẹ emwi hia gbe. Omwan ọkpa i sẹ gha ru emwi n'ọ khua vbe agbon.",
         "proverb_reasoning"),
        ("Translate into Bini-Edo: 'The King reigns with wisdom, truth, and justice for all people.'",
         "Vbe Bini-Edo: 'Ọba gha ya ẹwaẹn, ẹmwata, vbe ugbẹn-ẹmwẹn nẹ iran gbẹ gha sẹ ẹvbo.'",
         "translation"),
        ("Vb' a ya gbe ekom ke Edo vbe owie vbe ota?",
         "Vbe owie, a gha kọyọ: 'Ọbowie' ma ọ re 'Kọyọ rre'. Vbe ota, a gha kọyọ: 'Ọb'ota'. Vbe a gha kọyọ omwan n'ọ kpọlọ (Odionwere), a gha gbe erunmwon 'Kọyọ, Baba' ma ọ re 'Ọba gha t'o kpere'.",
         "conversation"),
        ("Ghe emwi n'o rre mbuk Igodomigodo vbe Benin.",
         "Igodomigodo ọ re erinmwi nẹ ẹvbo Benin n'eka kakaka gha re. Ogiso ọ re eniwan nẹ iran gha gbe erunmwon ye otọ na, nẹ Ọba gha re vbe iyeke nẹ iran.",
         "cultural_history")
    ]

    # General English & Reasoning Seeds (Anti-Catastrophic Forgetting)
    english_seeds = [
        ("Explain the concept of quantum computing and how qubits differ from classical bits.",
         "In classical computing, the fundamental unit of information is a bit, which exists in a definite state of either 0 or 1. In quantum computing, the fundamental unit is a qubit (quantum bit). Qubits leverage quantum mechanical principles: Superposition (existing in a linear combination of |0⟩ and |1⟩ simultaneously) and Entanglement (correlated states across multiple qubits). This allows quantum computers to process complex multidimensional state spaces exponentially faster for specific problems like cryptography and molecular simulation.",
         "stem_qa"),
        ("Write a Python function to perform binary search on a sorted list with complete docstrings.",
         "```python\ndef binary_search(arr: list[int], target: int) -> int:\n    \"\"\"Perform binary search on a sorted list.\n    \n    Args:\n        arr: A sorted list of integers.\n        target: The value to search for.\n        \n    Returns:\n        The index of target if found, else -1.\n    \"\"\"\n    low, high = 0, len(arr) - 1\n    while low <= high:\n        mid = (low + high) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            low = mid + 1\n        else:\n            high = mid - 1\n    return -1\n```",
         "coding_qa"),
        ("What is AfriWise and how does it handle African multilingual intelligence?",
         "AfriWise is a culturally authentic, high-capability multilingual AI system designed to understand and generate fluent Igbo, Efik/Ibibio, Bini/Edo, and English. Operating on the 'Translator of Knowledge' paradigm, it pairs deep global world knowledge with native African grammatical precision and cultural nuance.",
         "general_qa")
    ]

    for u, a, t in igbo_seeds:
        sp = get_distributed_system_prompt(rng)
        seed_items.append(make_chatml_record(u, a, sp, {"language": "igbo", "task_type": t}))

    for u, a, t in efik_seeds:
        sp = get_distributed_system_prompt(rng)
        seed_items.append(make_chatml_record(u, a, sp, {"language": "efik", "task_type": t}))

    for u, a, t in bini_seeds:
        sp = get_distributed_system_prompt(rng)
        seed_items.append(make_chatml_record(u, a, sp, {"language": "bini", "task_type": t}))

    for u, a, t in english_seeds:
        sp = get_distributed_system_prompt(rng)
        seed_items.append(make_chatml_record(u, a, sp, {"language": "english", "task_type": t}))

    return seed_items

# 6. Execute Multi-Source Ingestion Pipeline
rng = random.Random(3407)
raw_chatml_records = []
seen_hashes = set()

def add_chatml_record(user_text: str, asst_text: str, lang: str, task: str):
    u = sanitize_text(user_text)
    a = sanitize_text(asst_text)
    if is_clean_record(u) and is_clean_record(a):
        h = hashlib.sha256(f"{u}|{a}".encode("utf-8")).hexdigest()
        if h not in seen_hashes:
            seen_hashes.add(h)
            sp = get_distributed_system_prompt(rng)
            raw_chatml_records.append(make_chatml_record(u, a, sp, {"language": lang, "task_type": task}))

# Step A: Verified Seed Corpus Injection
seed_items = generate_verified_multilingual_seed_corpus(rng)
for item in seed_items:
    u = item["messages"][-2]["content"]
    a = item["messages"][-1]["content"]
    lang = item.get("metadata", {}).get("language", "multilingual")
    task = item.get("metadata", {}).get("task_type", "general")
    add_chatml_record(u, a, lang, task)

# Step B: Direct Ingestion from OPUS JW300 Parallel Corpora
for src, tgt, lang_label in [("en", "ig", "igbo"), ("en", "efi", "efik"), ("en", "bin", "bini")]:
    pairs = download_opus_jw300_pairs(src, tgt, max_pairs=5000)
    for en_text, tgt_text in pairs:
        add_chatml_record(f"Translate to {lang_label.capitalize()}: {en_text}", tgt_text, lang_label, "translation")
        add_chatml_record(f"Translate to English: {tgt_text}", en_text, lang_label, "translation")

# Step C: Direct Ingestion from Local / Google Drive Master Corpora
candidate_corpus_files = [
    DIR_DATA / "afriwise_master_corpus.jsonl",
    DIR_DATA / "afriwise_master_training_dataset.jsonl",
    DIR_DATA / "training_dataset_chatml.jsonl",
    Path("data/processed/afriwise_master_training_dataset.jsonl"),
    Path("data/processed/afriwise_v2_chatml.jsonl"),
    Path("data/export/huggingface/afriwise_master_corpus.jsonl")
]

for c_path in candidate_corpus_files:
    if c_path.exists():
        print(f"  ✓ Loading local / Drive corpus: {c_path}")
        try:
            with open(c_path, "r", encoding="utf-8", errors="replace") as f:
                for line in f:
                    if not line.strip():
                        continue
                    try:
                        obj = json.loads(line)
                        if "messages" in obj and len(obj["messages"]) >= 2:
                            u = obj["messages"][-2]["content"]
                            a = obj["messages"][-1]["content"]
                            lang = obj.get("metadata", {}).get("language", "multilingual")
                            add_chatml_record(u, a, lang, "conversational")
                        elif "text" in obj and "english" in obj:
                            lang = obj.get("language", "african")
                            add_chatml_record(f"Translate to English: {obj['text']}", obj["english"], lang, "translation")
                            add_chatml_record(f"Translate to {lang.capitalize()}: {obj['english']}", obj["text"], lang, "translation")
                        elif "instruction" in obj and "output" in obj:
                            add_chatml_record(obj["instruction"], obj["output"], "multilingual", "instruction")
                    except Exception:
                        pass
        except Exception as e:
            print(f"  ℹ️ Notice while reading {c_path}: {e}")

# 7. Persist Processed ChatML JSONL to Google Drive and Local Storage
PROCESSED_JSONL_PATH = DIR_DATA / "training_dataset_chatml.jsonl"
print(f"\n💾 Writing {len(raw_chatml_records):,} standardized ChatML records to {PROCESSED_JSONL_PATH}...")
with open(PROCESSED_JSONL_PATH, "w", encoding="utf-8") as f:
    for r in raw_chatml_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 8. Build Hugging Face Dataset with Tokenized ChatML Formatter
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list([{"messages": r["messages"]} for r in raw_chatml_records])
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✓ Hugging Face Dataset constructed successfully with {len(dataset):,} training samples.")
print(f"✓ Sample Tokenized Preview:\n{dataset[0]['text'][:300]}...\n")


## Cell 5: LoRA / QLoRA Adapter Configuration & Parameter Audit

Injects low-rank trainable matrices into all 7 linear projection layers of the transformer architecture:
- **Attention Projections**: `q_proj`, `k_proj`, `v_proj`, `o_proj` (Cross-lingual attention routing)
- **MLP Projections**: `gate_proj`, `up_proj`, `down_proj` (Lexical mapping and factual knowledge retention)

### LoRA Hyperparameters:
- Rank: $r = 16$
- Alpha: $\alpha = 32$ ($\alpha / r = 2.0$ scaling factor)
- Dropout: $0.0$ (Allows Unsloth fused kernel speedup)
- Gradient Checkpointing: Active (Reduces peak activation memory by ~30%)

In [ ]:
# ==============================================================================
# Cell 5: LoRA Adapter Injection & Parameter Audit
# ==============================================================================
print("=" * 75)
print("⚙️ Injecting LoRA Trainable Matrices across All Linear Projections")
print("=" * 75)

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

if UNSLOTH_AVAILABLE:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=TARGET_MODULES,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    from peft import LoraConfig, get_peft_model
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=TARGET_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

print("\n📊 LoRA Trainable Parameters Telemetry:")
model.print_trainable_parameters()


## Cell 6: SFTTrainer Execution, Live Loss Tracking & Google Drive Checkpointing

Executes Supervised Fine-Tuning (SFT) using Hugging Face `TRL` + `Transformers`.

### Hyperparameter Configuration:
- Micro Batch Size: `2`
- Gradient Accumulation Steps: `4` (Effective batch size = `8`)
- Learning Rate: `2e-4` with **Cosine Learning Rate Schedule**
- Optimizer: `adamw_8bit` (Paged 8-bit AdamW)
- Max Sequence Length: `2048` tokens
- Checkpointing: Step checkpoints saved to Google Drive every `60` steps

In [ ]:
# ==============================================================================
# Cell 6: SFTTrainer Fine-Tuning Loop & Checkpointing
# ==============================================================================
from trl import SFTTrainer
from transformers import TrainingArguments

print("=" * 75)
print("🚀 Starting AfriWise Multilingual SFT Training Loop")
print("=" * 75)

is_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=15,
    max_steps=120,
    learning_rate=2e-4,
    fp16=not is_bf16,
    bf16=is_bf16,
    logging_steps=10,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir=str(DIR_CHECKPOINTS),
    save_strategy="steps",
    save_steps=60,
    save_total_limit=2,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2 if os.cpu_count() and os.cpu_count() > 1 else 1,
    packing=False,
    args=training_args
)

if torch.cuda.is_available():
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 2)
    print(f"✓ Initial GPU Memory Reserved: {start_gpu_memory} GB")

trainer_stats = trainer.train()

if torch.cuda.is_available():
    peak_gpu_memory = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 2)
    total_vram = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
    print(f"✓ Peak GPU Memory Used: {peak_gpu_memory} GB / {total_vram} GB ({(peak_gpu_memory / total_vram) * 100:.1f}%)")

print(f"\n🎉 Training complete! Final Metrics:\n{trainer_stats.metrics}")


## Cell 7: Multilingual Inference Evaluation Suite & Interactive Widget

Evaluates the trained model across five cross-lingual test domains:
1. **English General Intelligence & STEM**: Tests non-catastrophic retention of physics/math/reasoning.
2. **Igbo Translation & Cultural Philosophy**: Tests proverbs and accurate translation.
3. **Efik/Ibibio Translation & Heritage**: Tests polite greetings and cultural definitions.
4. **Bini/Edo History & Royal Tradition**: Tests Oba of Benin historical comprehension.
5. **Multilingual Code-Switching**: Tests seamless inter-sentence language switching.

In [ ]:
# ==============================================================================
# Cell 7: Multilingual Inference Evaluation Suite & Interactive Widget
# ==============================================================================
if UNSLOTH_AVAILABLE:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

eval_test_suite = [
    {
        "domain": "1. English STEM & General Reasoning",
        "prompt": "Explain the concept of quantum superposition in simple terms with an everyday analogy."
    },
    {
        "domain": "2. Igbo Language & Cultural Wisdom",
        "prompt": "Translate 'Knowledge is wealth and brings peace to the community' into Igbo and explain the proverb 'Ilu bụ mmanụ e ji eri okwu'."
    },
    {
        "domain": "3. Efik/Ibibio Translation & Greetings",
        "prompt": "How do you say 'Good morning', 'How are you?', and 'Welcome to our home' in Efik/Ibibio?"
    },
    {
        "domain": "4. Bini/Edo History & Royal Heritage",
        "prompt": "Kọyọ! What is the historical significance of the Oba of Benin in ancient Edo civilization?"
    },
    {
        "domain": "5. Multilingual Code-Switching Dialogue",
        "prompt": "Nno! Good evening my friend. Kedụ ka mmemme taa siri gaa? Also tell me in Bini how to say thank you."
    }
]

print("=" * 75)
print("🌍 AfriWise Post-Training Multilingual Evaluation Suite")
print("=" * 75)

for test in eval_test_suite:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_KNOWLEDGE_TRANSLATOR},
        {"role": "user", "content": test["prompt"]}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    if torch.cuda.is_available():
        inputs = inputs.to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=250,
            temperature=0.3,
            top_p=0.9,
            use_cache=True
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n🧪 [Domain: {test['domain']}]")
    print(f"👤 User: {test['prompt']}")
    print(f"🤖 AfriWise:\n{response.strip()}")
    print("-" * 75)

def interactive_afriwise_chat(user_query: str, system_prompt: str = SYSTEM_PROMPT_KNOWLEDGE_TRANSLATOR) -> str:
    """Helper function for interactive testing in Colab."""
    msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_query}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    if torch.cuda.is_available():
        inp = inp.to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=inp, max_new_tokens=300, temperature=0.3, top_p=0.9, use_cache=True)
    return tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()

print("✓ Evaluation complete! You can test custom prompts using: interactive_afriwise_chat('your question')")


## Cell 8: 3-Way Model Export Suite & Ollama Deployment (R3)

Saves and exports the fine-tuned model directly to persistent Google Drive storage in three formats:
1. **LoRA Adapters (~150 MB)**: Lightweight PEFT adapter weights (`adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`).
2. **16-Bit Merged Hugging Face Model (~7.6 GB - 16 GB)**: Standalone FP16 weights ready for vLLM, Hugging Face Hub, or TGI.
3. **Direct GGUF Quantized Binary `Q4_K_M` (~2.2 GB - 4.8 GB)**: High-performance 4-bit quantized binary for local Ollama, LM Studio, or `llama.cpp` deployment.
4. **Generated Ollama `Modelfile`**: Production-ready Ollama configuration file.

In [ ]:
# ==============================================================================
# Cell 8: 3-Way Model Export Suite & Ollama Modelfile
# ==============================================================================
import os
import shutil
from pathlib import Path

LORA_EXPORT_DIR = DIR_EXPORT / "afriwise_lora"
MERGED_EXPORT_DIR = DIR_EXPORT / "afriwise_merged_16bit"
GGUF_EXPORT_DIR = DIR_EXPORT / "afriwise_gguf_q4"

# Clean directories to prevent "cannot mmap an empty file" errors from interrupted runs
for d in [LORA_EXPORT_DIR, MERGED_EXPORT_DIR, GGUF_EXPORT_DIR]:
    if d.exists():
        shutil.rmtree(d, ignore_errors=True)

print("=" * 75)
print("💾 AfriWise 3-Way Model Export Suite")
print("=" * 75)

# 1. Export LoRA Adapter
print(f"\n[Export 1/3] Saving LoRA adapter checkpoint to: {LORA_EXPORT_DIR}...")
model.save_pretrained(str(LORA_EXPORT_DIR))
tokenizer.save_pretrained(str(LORA_EXPORT_DIR))
print("✓ LoRA adapter saved successfully.")

# 2. Export 16-Bit Standalone Merged Model
print(f"\n[Export 2/3] Merging LoRA weights into 16-bit standalone model at: {MERGED_EXPORT_DIR}...")
if UNSLOTH_AVAILABLE:
    try:
        model.save_pretrained_merged(str(MERGED_EXPORT_DIR), tokenizer, save_method="merged_16bit")
        print("✓ 16-bit merged model exported via Unsloth!")
    except Exception as e:
        print(f"ℹ️ Unsloth merged export note: {e}. Trying standard PEFT merge...")
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(str(MERGED_EXPORT_DIR))
        tokenizer.save_pretrained(str(MERGED_EXPORT_DIR))
else:
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(MERGED_EXPORT_DIR))
    tokenizer.save_pretrained(str(MERGED_EXPORT_DIR))
    print("✓ 16-bit merged model exported via PEFT merge_and_unload()!")

# 3. Direct GGUF Q4_K_M Quantization Export
print(f"\n[Export 3/3] Quantizing and exporting GGUF Q4_K_M binary to: {GGUF_EXPORT_DIR}...")
if UNSLOTH_AVAILABLE:
    try:
        model.save_pretrained_gguf(str(GGUF_EXPORT_DIR), tokenizer, quantization_method="q4_k_m")
        print("✓ GGUF Q4_K_M binary exported successfully via Unsloth!")
    except Exception as e:
        print(f"ℹ️ Unsloth direct GGUF export note: {e}. Standalone 16-bit model can be quantized with llama.cpp or convert_hf_to_gguf.py.")
else:
    print("ℹ️ Standalone 16-bit weights exported. Use llama.cpp / convert_hf_to_gguf.py to quantize into GGUF.")

# 4. Generate Production Ollama Modelfile
MODELFILE_PATH = DIR_EXPORT / "Modelfile"
modelfile_lines = [
    "FROM ./afriwise_q4_k_m.gguf",
    "",
    "# Inference hyperparameters",
    "PARAMETER temperature 0.3",
    "PARAMETER top_p 0.9",
    "PARAMETER repeat_penalty 1.1",
    "PARAMETER num_ctx 4096",
    "",
    "# ChatML stop tokens",
    'PARAMETER stop "<|im_end|>" ',
    'PARAMETER stop "<|endoftext|>" ',
    'PARAMETER stop "<|end|>" ',
    "",
    "# ChatML prompt template",
    'TEMPLATE """{{ if .System }}<|im_start|>system',
    "{{ .System }}<|im_end|>",
    "{{ end }}{{ if .Prompt }}<|im_start|>user",
    "{{ .Prompt }}<|im_end|>",
    '{{ end }}<|im_start|>assistant',
    '{{ .Response }}<|im_end|>"""',
    "",
    '# "Translator of Knowledge" unrestrictive multilingual persona',
    'SYSTEM """You are AfriWise - a highly capable, knowledgeable AI. When asked a question in Igbo, Efik/Ibibio, or Bini/Edo, you access your vast general knowledge and output the answer flawlessly in that specific language. You also converse fluently and intelligently in English."""'
]
with open(MODELFILE_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(modelfile_lines) + "\n")

print(f"\n✓ Generated Ollama Modelfile at: {MODELFILE_PATH}")
print("\n" + "=" * 75)
print(f"🎉 AfriWise Colab Retraining Pipeline Complete! All artifacts preserved in: {DIR_EXPORT}")
print("=" * 75)
